In [2]:
import glob, os, pysam
import multiprocessing as mp
import pandas as pd

In [3]:
def merge_bams(infiles, outfile, min_reads=2, rmdup=True):
    fw = None
    tmpfile = outfile + ".unsorted.bam"
    statfile = outfile[:-4] + ".flagstat"
    for i, infile in enumerate(infiles):
        # print("%s/%s\t%s" % (i, len(infiles), infile))
        with pysam.AlignmentFile(infile) as f:
            if fw is None:
                fw = pysam.AlignmentFile(tmpfile, "wb", f)
            for s in f:
                if s.get_tag("CS") < min_reads:
                    continue
                if rmdup and s.is_duplicate:
                    continue
                fw.write(s)
    fw.close()
    ! samtools sort -@ 8 -o {outfile} {tmpfile}
    ! samtools index -@ 8 {outfile}
    ! rm {tmpfile}
    ! samtools flagstat -@ 8 {outfile} > {statfile}

In [6]:
pool = mp.Pool(8)
for path in sorted(glob.glob("results/tables/*.csv")):
    name = path.split("/")[-1][:-4]
    outfile = "results/bams/%s.bam" % name
    if os.path.exists(outfile):
        print("%s exists!" % outfile)
        continue
    d = pd.read_csv(path, index_col=0)
    bamfiles = ["../../1_NanoNASCseq/results/3_mapping/6_remove_duplicate/%s/%s.bam" % (cell.split(".")[0], cell) for cell in d.index]
    pool.apply_async(merge_bams, (bamfiles, outfile))
pool.close()
pool.join()

results/bams/K562.bam exists!
results/bams/K562.s4U_0uM_3h.bam exists!
results/bams/K562.s4U_100uM_3h.bam exists!
results/bams/K562.s4U_200uM_3h.bam exists!
results/bams/K562.s4U_400uM_3h.bam exists!
results/bams/K562.s4U_500uM_3h.bam exists!
results/bams/K562.s4U_50uM_15min.bam exists!
results/bams/K562.s4U_50uM_1h.bam exists!
results/bams/K562.s4U_50uM_2h.bam exists!
results/bams/K562.s4U_50uM_30min.bam exists!
results/bams/K562.s4U_50uM_3h.bam exists!
results/bams/K562.s4U_50uM_3h.highTC.bam exists!
results/bams/K562_FUCCI_G1.s4U_0uM_3h.bam exists!
results/bams/K562_FUCCI_G1.s4U_50uM_2h.bam exists!
results/bams/K562_FUCCI_G1.s4U_50uM_3h.bam exists!
results/bams/K562_FUCCI_G2M.s4U_0uM_3h.bam exists!
results/bams/K562_FUCCI_G2M.s4U_50uM_3h.bam exists!
results/bams/K562_FUCCI_S1.s4U_0uM_3h.bam exists!
results/bams/K562_FUCCI_S1.s4U_50uM_3h.bam exists!
results/bams/K562_FUCCI_S2.s4U_0uM_3h.bam exists!
results/bams/K562_FUCCI_S2.s4U_50uM_3h.bam exists!
results/bams/MouseBlastocyst_EPI.s4